# Método Delta
Esse método serve para responder:

> Se eu sei como uma estimativa se comporta (tipo média, variância assintótica), o que acontece quando eu aplico uma função nela?

O método diz que, se eu tenho um estimador $\hat{\theta}$ tal que $\sqrt{n}(\hat{\theta} - \theta) \overset{d}{\to} \cal{N}(0,\sigma^2)$, então o método diz que, dado uma função $g$ com $g'(\theta) \neq 0$:
$$
    \sqrt{n} \frac{ (g(\hat{\theta}) - g(\theta)) }{ \sigma |g'(\theta)| } \overset{d}{\to} \cal{N}(0,1)
$$

In [83]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
import seaborn as sns

Vamos imaginar o cenário onde observamos uma amostra $X_1,...,X_n \sim Poisson(\lambda)$, mas não queremos estimar $\lambda$, mas sim $\sqrt{\lambda}$ (É útil pois aplicar essa transformação estabiliza a variação da amostra). Sabemos que um bom estimador de $\lambda$ é $\overline{X}_n$, mas e para $\sqrt{\lambda}$? Se estivermos estimando por máxima verossimilhança, vai ser $\sqrt{\overline{X}_n}$, mas se quisermos fazer um intervalo de confiança, **precisamos descobrir a distribuição do estimador de $\sqrt{\lambda}$**, o que não é trivial, mas o **método delta resolve**

In [84]:
lam = 4  # Fingimos que não sabemos o valor de lambda e queremos estimá-lo a partir dos dados
n = 10000  # tamanho da amostra

# Gerar amostra de dados a partir da distribuição de Poisson
data = np.random.poisson(lam, n)

In [85]:
data

array([2, 2, 6, ..., 5, 1, 4], shape=(10000,), dtype=int32)

De acordo com o método delta, temos que:
$$
    g(\lambda) = \sqrt{\lambda} \Rightarrow g'(\lambda) = \frac{1}{2\sqrt{\lambda}}
$$
$$
    2 \sqrt{n} (\sqrt{\hat{\lambda}} - \sqrt{\lambda}) \overset{d}{\to} \cal{N}(0,1)
$$
logo, podemos gerar um intervalo de confiança para $\sqrt{\lambda}$:
$$
    \mathbb{P}(\sqrt{\hat{\lambda}} \in (A, B)) \approx \gamma      \\

    A = \sqrt{\hat{\lambda}} - z_{\frac{1+\gamma}{2}} \frac{1}{2\sqrt{n \hat{\lambda}}}     \\
    B = \sqrt{\hat{\lambda}} + z_{\frac{1+\gamma}{2}} \frac{1}{2\sqrt{n \hat{\lambda}}}     \\

    \gamma \in (0,1)
$$
onde $z_\alpha$ é o $\alpha$ quantil da $\cal{N}(0,1)$

In [86]:
gamma = 0.97
lambd_estimada = np.mean(data)
A = np.sqrt(lambd_estimada) - norm.ppf(loc=0, scale=1, q=(1 + gamma) / 2) / (2 * np.sqrt(lambd_estimada * n))
B = np.sqrt(lambd_estimada) + norm.ppf(loc=0, scale=1, q=(1 + gamma) / 2) / (2 * np.sqrt(lambd_estimada * n))

In [87]:
print("Intervalo de confiança para sqrt(lambda): [{:.4f}, {:.4f}]".format(A, B))

Intervalo de confiança para sqrt(lambda): [1.9961, 2.0070]


porém, vamos ver o que acontece quando geramos o intervalo de confiança padrão de $\lambda$ e aplicamos a transformação em ambas as laterais:
$$
    \mathbb{P}(\sqrt{\hat{\lambda}} \in (A, B)) \approx \gamma      \\

    A = \sqrt{\hat{\lambda}} - z_{\frac{1+\gamma}{2}} \sqrt{\frac{\hat{\lambda}}{n}}     \\
    B = \sqrt{\hat{\lambda}} + z_{\frac{1+\gamma}{2}} \sqrt{\frac{\hat{\lambda}}{n}}     \\

    \gamma \in (0,1)
$$

In [88]:
gamma = 0.97
lambd_estimada = np.mean(data)
A = lambd_estimada - norm.ppf(loc=0, scale=1, q=(1 + gamma) / 2) * np.sqrt(lambd_estimada / n)
B = lambd_estimada + norm.ppf(loc=0, scale=1, q=(1 + gamma) / 2) * np.sqrt(lambd_estimada / n)

In [89]:
print(f"Intervalo de confiança para lambda: [{A:.4f}, {B:.4f}]")
print(f"Intervalo de confiança para sqrt(lambda) transformado: [{np.sqrt(A):.4f}, {np.sqrt(B):.4f}]")

Intervalo de confiança para lambda: [3.9628, 4.0496]
Intervalo de confiança para sqrt(lambda) transformado: [1.9907, 2.0124]


Nesse caso em específico, aplicar a transformação $g$ no intervalo de confiança de $\lambda$ da uma boa estimativa, porém, nem sempre isso é possível! Esse exemplo foi um exemplo em que o método delta não era tão necessário, mas e se tivéssemos o seguinte problema?:

> Suponha que temos $X_1,...,X_n \sim \cal{N} (\mu_X,\sigma^2)$ e $Y_1,...,Y_m \sim \cal{N} (\mu_Y,\sigma^2)$, como podemos estimar $g(\mu_X,\mu_Y) = \frac{\mu_X}{\mu_Y}?$

nós até sabemos que $\overline{X}_n \sim \cal{N}(\mu_X,\sigma^2/n)$ e $\overline{Y}_m \sim \cal{N}(\mu_Y,\sigma^2/m)$, porém a distribuição de
$$
    \frac{\overline{X}_n}{\overline{Y}_m}
$$
é muito esquisita e não tem intervalo fácil para transformar, então **precisamos** usar o método delta.

1. **Achando $\nabla g$**:
$$
    \nabla g = (\frac{1}{\mu_Y}, -\frac{\mu_X}{\mu_Y ^2})
$$

2. **Variância por método delta**
$$
    \mathbb{V}[g(\hat{\mu_X}, \hat{\mu_Y})] \approx \nabla g^T \Sigma \nabla g
$$
como as amostras são independentes:
$$
    \Sigma = \left[\begin{matrix}
        \sigma^2/n, 0   \\
        0, \sigma^2/m
    \end{matrix}\right]
$$
Logo:
$$
    \mathbb{V}[g(\mu_X, \mu_Y)] \approx \frac{\sigma^2}{n \mu_Y^2} + \frac{\mu_X^2 \sigma^2}{m \mu_Y^4}
$$

3. **Gerar intervalo de confiança**
$$
    \frac{\overline{X}_n}{\overline{Y}_m} \pm z_{\frac{1 + \gamma}{2}} \sqrt{\mathbb{V}[g(\hat{\mu_X}, \hat{\mu_Y})]}
$$

Para facilitar, assumimos que a variância é conhecida

In [90]:
n = 1000
m = 5000

mu_x = 5
mu_y = 10

var = 9

sample_x = np.random.normal(mu_x, np.sqrt(var), n)
sample_y = np.random.normal(mu_y, np.sqrt(var), m)

In [91]:
print(f"Razão original (não temos conhecimento): {mu_x/mu_y}")

Razão original (não temos conhecimento): 0.5


In [92]:
mu_x_estimada = np.mean(sample_x)
mu_y_estimada = np.mean(sample_y)

var_estimada = var / (n * mu_y_estimada**2) + (var * mu_x_estimada**2) / (m * mu_y_estimada**4)

In [93]:
gamma = 0.97
A = mu_x_estimada / mu_y_estimada - norm.ppf(loc=0, scale=1, q=(1 + gamma) / 2) * np.sqrt(var_estimada)
B = mu_x_estimada / mu_y_estimada + norm.ppf(loc=0, scale=1, q=(1 + gamma) / 2) * np.sqrt(var_estimada)

In [94]:
print(f"Intervalo de confiança para mu_x/mu_y: [{A:.4f}, {B:.4f}]")

Intervalo de confiança para mu_x/mu_y: [0.4772, 0.5193]
